# Exercise 1 — From Pixels to Visual Evidence — Sample Solution

**Computer Vision for Industrial Systems**

This is the teacher/sample solution version. It contains one possible implementation for all TODOs.

In [ ]:
from pathlib import Path
import sys
import math
import os
# Ensure Matplotlib uses a writable cache directory in Binder/sandbox environments.
os.environ.setdefault("MPLCONFIGDIR", str((Path.cwd() / ".matplotlib_cache").resolve()))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo_root = Path.cwd().parents[1] if Path.cwd().name == "solutions" else (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.append(str(repo_root / "src"))

from cvis_utils import (
    load_gray, show_images, plot_histograms,
    mm_per_pixel, pixels_per_feature,
    downsample_image, motion_blur_kernel, apply_motion_blur,
    add_gaussian_noise, gamma_correct, quantize_image,
    sobel_energy, laplacian_variance,
    otsu_segment, clean_mask, region_table, iou
)

DATA = repo_root / "data"
SYN = DATA / "synthetic"
IND = DATA / "industrial_like"
print("Repository root:", repo_root)

## 0. Load and inspect images

In [ ]:
checker = load_gray(SYN / "checkerboard_highres.png")
brick = load_gray(SYN / "brick_wall_synthetic.png")
surface = load_gray(IND / "images" / "surface_defect_bright_scratch_easy.png")
mask_gt = load_gray(IND / "masks" / "surface_defect_bright_scratch_easy_mask.png") > 0.5

show_images(
    [checker, brick, surface, mask_gt],
    ["checkerboard", "brick-like texture", "surface defect", "ground-truth mask"],
    ncols=4,
    figsize=(14, 4),
)

## Part A — Object-space resolution, sampling, and aliasing

In [ ]:
def student_mm_per_pixel(fov_mm: float, n_pixels: int) -> float:
    return float(fov_mm) / float(n_pixels)


def student_pixels_per_feature(feature_size_mm: float, mm_per_px: float) -> float:
    return float(feature_size_mm) / float(mm_per_px)


fov_mm = 400.0
n_px = 2448
defect_size_mm = 0.5

mm_px = student_mm_per_pixel(fov_mm, n_px)
px_defect = student_pixels_per_feature(defect_size_mm, mm_px)

print(f"Object-space resolution: {mm_px:.4f} mm/px")
print(f"0.5 mm defect spans: {px_defect:.2f} px")

In [ ]:
answer_A1 = """
The object-space resolution is about 0.163 mm/px, so a 0.5 mm defect spans about 3.1 pixels.
This may be enough for coarse detection under good contrast, but it is marginal for reliable localization or measurement.
Pixel count is not sufficient: contrast, optical blur, motion blur, focus, noise, and illumination stability also matter.
"""
print(answer_A1)

In [ ]:
scale = 0.18

brick_bad = downsample_image(brick, scale=scale, anti_aliasing=False)
brick_good = downsample_image(brick, scale=scale, anti_aliasing=True)

show_images(
    [brick, brick_bad, brick_good],
    ["original", "downsampled, no anti-aliasing", "downsampled, anti-aliasing"],
    ncols=3,
    figsize=(13, 4),
)

In [ ]:
row_original = brick.shape[0] // 2
row_bad = brick_bad.shape[0] // 2
row_good = brick_good.shape[0] // 2

plt.figure(figsize=(12, 4))
plt.plot(brick[row_original, :], label="original", alpha=0.8)
plt.plot(np.linspace(0, brick.shape[1]-1, brick_bad.shape[1]), brick_bad[row_bad, :], label="downsampled, no AA", alpha=0.8)
plt.plot(np.linspace(0, brick.shape[1]-1, brick_good.shape[1]), brick_good[row_good, :], label="downsampled, AA", alpha=0.8)
plt.title("Intensity profiles")
plt.xlabel("column index")
plt.ylabel("intensity")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Part B — Blur, noise, saturation, bit depth, and gamma

In [ ]:
surface_blur = apply_motion_blur(surface, length=21, angle_deg=0)

show_images(
    [surface, surface_blur],
    ["original surface defect", "motion-blurred"],
    ncols=2,
    figsize=(10, 4),
)

print("Sobel energy original:", sobel_energy(surface))
print("Sobel energy blurred :", sobel_energy(surface_blur))
print("Laplacian variance original:", laplacian_variance(surface))
print("Laplacian variance blurred :", laplacian_variance(surface_blur))

In [ ]:
surface_noisy = add_gaussian_noise(surface, sigma=0.08, seed=123)
surface_clipped = np.clip(surface * 1.7, 0, 1)
surface_quantized = quantize_image(surface, bits=3)

show_images(
    [surface, surface_noisy, surface_clipped, surface_quantized],
    ["original", "noisy", "clipped / saturated", "3-bit quantized"],
    ncols=4,
    figsize=(14, 4),
)

plot_histograms(
    [surface, surface_noisy, surface_clipped, surface_quantized],
    ["original", "noisy", "clipped", "quantized"],
    bins=64,
    figsize=(14, 3),
)

In [ ]:
gamma = 2.2
surface_gamma = gamma_correct(surface, gamma=gamma)

show_images([surface, surface_gamma], ["original / linear-like", f"gamma corrected, gamma={gamma}"], ncols=2, figsize=(10, 4))
plot_histograms([surface, surface_gamma], ["original", "gamma corrected"], bins=64, figsize=(10, 3))

In [ ]:
answer_B = """
Motion blur reduced edge strength and made the scratch boundary less reliable.
Noise made weak texture and defect evidence less stable, while clipping destroyed information in bright regions.
Gamma correction changed the intensity distribution and may make the image look different for display, but it changes the meaning of intensity values for measurement.
Before trying a model, I would improve illumination, reduce exposure time if motion is present, avoid saturation, and keep preprocessing consistent.
"""
print(answer_B)

## Part C — Thresholding, morphology, and simple measurement

In [ ]:
mask_raw, threshold_value = otsu_segment(surface, foreground="bright")

print("Otsu threshold:", threshold_value)
show_images([surface, mask_raw], ["surface image", "raw threshold mask"], ncols=2, figsize=(10, 4))

In [ ]:
mask_clean = clean_mask(mask_raw, min_size=80, closing_radius=2)

show_images([mask_raw, mask_clean, mask_gt], ["raw mask", "cleaned mask", "ground truth mask"], ncols=3, figsize=(12, 4))
print("IoU raw mask   :", iou(mask_raw, mask_gt))
print("IoU clean mask :", iou(mask_clean, mask_gt))

In [ ]:
regions = region_table(mask_clean, mm_per_px=mm_px)
df_regions = pd.DataFrame(regions)
df_regions

In [ ]:
hard_image_path = IND / "images" / "surface_defect_dark_scratch_01.png"
hard_mask_path = IND / "masks" / "surface_defect_dark_scratch_01_mask.png"

hard_img = load_gray(hard_image_path)
hard_gt = load_gray(hard_mask_path) > 0.5

hard_raw, hard_thr = otsu_segment(hard_img, foreground="dark")
hard_clean = clean_mask(hard_raw, min_size=100, closing_radius=2)

show_images([hard_img, hard_raw, hard_clean, hard_gt], ["hard image", "raw mask", "clean mask", "ground truth"], ncols=4, figsize=(14, 4))
print("Hard image threshold:", hard_thr)
print("Hard image IoU:", iou(hard_clean, hard_gt))

In [ ]:
final_recommendation = """
1. Sampling / resolution:
The 0.5 mm defect is only about 3 pixels wide in the running scenario, so this is marginal for localization and measurement.

2. Blur / noise / saturation / gamma:
Blur and saturation are especially dangerous because they destroy or clip information. Gamma changes intensity meaning and should be controlled if measurement is involved.

3. Thresholding / segmentation:
Simple thresholding can work when the defect is visually separable, but it is fragile when contrast changes, background texture is strong, or the defect has different polarity.

4. Imaging setup recommendation before ML:
I would first improve contrast with controlled illumination, ensure enough pixels per defect, reduce motion blur with shorter exposure or strobing, avoid saturation, and define a stable preprocessing pipeline.
"""
print(final_recommendation)

## Optional extension solution ideas

- Sweep motion blur length and plot edge metrics.
- Evaluate several defect images and compare IoU.
- Test foreground polarity automatically.
- Add adaptive thresholding.
- Estimate maximum exposure time for a chosen blur tolerance.